In [5]:
import pandas as pd

import random



# Roles and Skills Matrix

role_matrix = {

    "Data Scientist": {"oxygen": ["python", "machine learning", "statistics"], "secondary": ["sql", "tableau", "deep learning", "pandas", "nlp", "aws"]},

    "Data Analyst": {"oxygen": ["sql", "excel", "powerbi"], "secondary": ["python", "statistics", "tableau", "data visualization", "r"]},

    "ML Engineer": {"oxygen": ["python", "tensorflow", "mlops"], "secondary": ["pytorch", "aws", "docker", "kubernetes", "nlp"]},

    "Full Stack Dev": {"oxygen": ["react", "nodejs", "mongodb"], "secondary": ["express", "javascript", "html", "css", "git"]},

    "Backend Dev": {"oxygen": ["python", "django", "postgresql"], "secondary": ["flask", "redis", "docker", "api design", "aws"]},

    "Frontend Dev": {"oxygen": ["javascript", "react", "tailwind"], "secondary": ["html", "css", "nextjs", "vue", "typescript"]},

    "App Developer": {"oxygen": ["flutter", "dart", "firebase"], "secondary": ["kotlin", "swift", "react native", "sqlite", "ui/ux"]},

    "DevOps Engineer": {"oxygen": ["docker", "kubernetes", "linux"], "secondary": ["jenkins", "terraform", "aws", "ansible", "git"]},

    "Cloud Architect": {"oxygen": ["aws", "networking", "iam"], "secondary": ["azure", "gcp", "terraform", "python", "security"]},

    "Data Engineer": {"oxygen": ["spark", "sql", "etl"], "secondary": ["hadoop", "kafka", "airflow", "redshift", "python"]},

    "Cyber Security": {"oxygen": ["networking", "linux", "ethical hacking"], "secondary": ["wireshark", "metasploit", "siem", "firewalls", "cryptography"]},

    "Ethical Hacker": {"oxygen": ["web security", "python", "linux"], "secondary": ["burpsuite", "nmap", "vulnerability assessment", "owasp", "metasploit"]},

    "AI Researcher": {"oxygen": ["pytorch", "calculus", "research papers"], "secondary": ["python", "cuda", "nlp", "computer vision", "maths"]},

    "SOC Analyst": {"oxygen": ["incident response", "splunk", "log analysis"], "secondary": ["siem", "networking", "firewalls", "threat hunting", "linux"]},

    "IT Support": {"oxygen": ["hardware", "troubleshooting", "os"], "secondary": ["active directory", "office 365", "networking", "linux", "windows server"]}

}



qual_options = ["12th", "Diploma", "BCA", "BSc", "BTech", "MCA", "MTech"]

domains = list(role_matrix.keys()) + ["Sales", "Marketing", "HR", "Finance", "None (Fresher)"]



data = []



for i in range(30000):

    applied_role = random.choice(list(role_matrix.keys()))

    exp_years = random.randint(0, 15)



    # Past Domain Logic

    if exp_years == 0:

        past_domain = "None (Fresher)"

    else:

        past_domain = random.choice(domains)



    qualification = random.choice(qual_options)



    # Achievement Logic for Freshers

    has_internship = random.choice([0, 1])

    has_projects = random.choice([0, 1])



    # Skill Picking

    oxy = role_matrix[applied_role]["oxygen"]

    sec = role_matrix[applied_role]["secondary"]

    num_oxy = random.randint(0, 3)

    num_sec = random.randint(0, 5)

    skills_to_pick = random.sample(oxy, num_oxy) + random.sample(sec, num_sec)



    # --- Selection Scoring Logic ---

    score = 0

    score += (num_oxy * 25) # Max 75 (Oxygen skills are heavy)

    score += (num_sec * 5)   # Max 25



    # Experience vs Achievement Balance

    if exp_years > 0:

        # Domain Penalty/Bonus

        if applied_role == past_domain:

            score += (exp_years * 5) # Strong bonus for relevant exp

        else:

            score += (exp_years * 1) # Penalty Box: Low bonus for mismatch

    else:

        # Fresher Special Score

        if has_internship: score += 15

        if has_projects: score += 10



    # Qualification Weight

    qual_weight = {"12th": 5, "Diploma": 10, "BCA": 15, "BSc": 15, "BTech": 20, "MCA": 25, "MTech": 25}

    score += qual_weight[qualification]



    # HR Decision (Threshold 85)

    selected = 1 if score >= 85 else 0



    skills_str = ", ".join(skills_to_pick) if skills_to_pick else "none"

    data.append([applied_role, past_domain, exp_years, qualification, skills_str, has_internship, has_projects, selected])

In [6]:
df = pd.DataFrame(data, columns=['applied_role', 'past_domain', 'exp_years', 'qualification', 'skills', 'internship', 'projects', 'selected'])

df.to_csv("master_recruitment_data_v8.csv", index=False)

print("30,000 Rows Generated! Ready for Training. ✅")

30,000 Rows Generated! Ready for Training. ✅


In [7]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

# Load Data
df = pd.read_csv("master_recruitment_data_v8.csv")
X = df.drop('selected', axis=1)
y = df['selected']

# Preprocessing
qual_order = ["12th", "Diploma", "BCA", "BSc", "BTech", "MCA", "MTech"]

preprocessor = ColumnTransformer(
    transformers=[
        ('skills_tfidf', TfidfVectorizer(ngram_range=(1,1)), 'skills'),
        ('qual_ordinal', OrdinalEncoder(categories=[qual_order]), ['qualification']),
        ('cat_features', OneHotEncoder(handle_unknown='ignore'), ['applied_role', 'past_domain']),
    ], remainder='passthrough') # Remaining: exp_years, internship, projects

# Model Pipeline with XGBoost
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42))
])

In [8]:
# Train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model_pipeline.fit(X_train, y_train)

# Save
with open('job_model_v8.pkl', 'wb') as f:
    pickle.dump(model_pipeline, f)

print(f"Model v8 Trained! Accuracy: {model_pipeline.score(X_test, y_test)*100:.2f}% ✅")

Model v8 Trained! Accuracy: 94.80% ✅
